In [1]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if "adapter_config.json" in str(p) or "adapter_model.safetensors" in str(p):
        print(p)

/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/adapter_model.safetensors
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/adapter_config.json


In [2]:
import os

print(os.listdir("/kaggle/input/datasets/chenjunjiejaywang/"))

['model-dataset', 'model-parameters']


In [3]:
from pathlib import Path

root = Path("/kaggle/input/datasets/chenjunjiejaywang/model-parameters")
for p in root.rglob("*"):
    print(p)

/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/adapter_model.safetensors
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/adapter_config.json
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/README.md
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/tokenizer.json
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/tokenizer_config.json
/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter/chat_template.jinja


In [4]:
from pathlib import Path

root = Path("/kaggle/input/datasets/chenjunjiejaywang/model-dataset")
for p in root.rglob("*"):
    print(p)

/kaggle/input/datasets/chenjunjiejaywang/model-dataset/model.safetensors.index.json
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/config.json
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/model-00001-of-00002.safetensors
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/model-00002-of-00002.safetensors
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/tokenizer.json
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/tokenizer_config.json
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/chat_template.jinja
/kaggle/input/datasets/chenjunjiejaywang/model-dataset/generation_config.json


In [5]:
from pathlib import Path

ADAPTER_DIR = Path("/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter")

print("ADAPTER_DIR =", ADAPTER_DIR)
print("exists =", ADAPTER_DIR.exists())

for p in ADAPTER_DIR.iterdir():
    print(p.name)

ADAPTER_DIR = /kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter
exists = True
adapter_model.safetensors
adapter_config.json
README.md
tokenizer.json
tokenizer_config.json
chat_template.jinja


In [6]:
# ===== CELL 1: CONFIG + IMPORTS =====
import os
import re
import gc
import json
import math
import time
import random
import warnings
from io import BytesIO
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -----------------------------
# MODE SWITCH
# -----------------------------
# Kaggle 最终离线推理：必须用本地 base model + 本地 adapter
FINAL_OFFLINE = True

# 仅保留作参考；FINAL_OFFLINE=True 时不会用这个
BASE_MODEL_HUB_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# ===== 你当前 Kaggle 的真实路径 =====
LOCAL_BASE_MODEL_DIR = "/kaggle/input/datasets/chenjunjiejaywang/model-dataset"
ADAPTER_DIR = Path("/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter")

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
RUN_NAME = "svg_kaggle_infer"

def find_comp_dir():
    candidates = []
    for p in INPUT_ROOT.rglob("test.csv"):
        parent = p.parent
        if (parent / "sample_submission.csv").exists():
            candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(
            "Could not find competition data under /kaggle/input/ "
            "(need both test.csv and sample_submission.csv)."
        )
    candidates = sorted(set(candidates), key=lambda x: len(str(x)))
    return candidates[0]

# ===== 基础检查 =====
base_dir = Path(LOCAL_BASE_MODEL_DIR)
if not base_dir.exists():
    raise FileNotFoundError(f"Base model dir not found: {base_dir}")

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"Adapter dir not found: {ADAPTER_DIR}")

required_base_files = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]
for fname in required_base_files:
    if not (base_dir / fname).exists():
        raise FileNotFoundError(f"Missing base model file: {base_dir / fname}")

has_base_weights = (
    (base_dir / "model.safetensors").exists()
    or (base_dir / "model.safetensors.index.json").exists()
    or len(list(base_dir.glob("model-*.safetensors"))) > 0
    or (base_dir / "pytorch_model.bin").exists()
)
if not has_base_weights:
    raise FileNotFoundError(f"No base model weights found under: {base_dir}")

required_adapter_files = [
    "adapter_config.json",
]
for fname in required_adapter_files:
    if not (ADAPTER_DIR / fname).exists():
        raise FileNotFoundError(f"Missing adapter file: {ADAPTER_DIR / fname}")

has_adapter_weights = (
    (ADAPTER_DIR / "adapter_model.safetensors").exists()
    or (ADAPTER_DIR / "adapter_model.bin").exists()
)
if not has_adapter_weights:
    raise FileNotFoundError(f"No adapter weights found under: {ADAPTER_DIR}")

COMP_DIR = find_comp_dir()

if FINAL_OFFLINE:
    BASE_MODEL_REF = LOCAL_BASE_MODEL_DIR
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["HF_HUB_OFFLINE"] = "1"
else:
    BASE_MODEL_REF = BASE_MODEL_HUB_ID

CONFIG = {
    "base_model": BASE_MODEL_REF,
    "test_path": str(COMP_DIR / "test.csv"),
    "sample_submission_path": str(COMP_DIR / "sample_submission.csv"),
    "work_dir": str(WORK_DIR),
    "adapter_dir": str(ADAPTER_DIR),

    "max_seq_length": 2048,
    "max_svg_chars_hard": 16000,   # 如果你最终确认规则是16000，可改成16000
    "svg_round_decimals": 2,

    "gen_max_new_tokens": 384,
    "infer_batch_size": 12,
    "repetition_penalty": 1.02,
}

SYSTEM_PROMPT = (
    "Generate only valid SVG markup for a simple icon or symbol that matches the user's request. "
    "Use exactly one root <svg xmlns='http://www.w3.org/2000/svg'> element. "
    "Return SVG only. No markdown. No explanation. No comments. No chat tags. "
    "Keep the SVG compact, semantically faithful, and visually centered."
)

print("Competition data:", COMP_DIR)
print("Base model dir:", LOCAL_BASE_MODEL_DIR)
print("Adapter dir:", ADAPTER_DIR)
print("Base model ref:", CONFIG["base_model"])
print(json.dumps(CONFIG, indent=2))

GPU: Tesla T4
Competition data: /kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline
Base model dir: /kaggle/input/datasets/chenjunjiejaywang/model-dataset
Adapter dir: /kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter
Base model ref: /kaggle/input/datasets/chenjunjiejaywang/model-dataset
{
  "base_model": "/kaggle/input/datasets/chenjunjiejaywang/model-dataset",
  "test_path": "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/test.csv",
  "sample_submission_path": "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/sample_submission.csv",
  "work_dir": "/kaggle/working",
  "adapter_dir": "/kaggle/input/datasets/chenjunjiejaywang/model-parameters/adapter",
  "max_seq_length": 2048,
  "max_svg_chars_hard": 16000,
  "svg_round_decimals": 2,
  "gen_max_new_tokens": 384,
  "infer_batch_size": 12,
  "repetition_penalty": 1.02
}


In [7]:
# ===== CELL 2: PATH CHECK =====
print("ADAPTER_DIR exists:", ADAPTER_DIR.exists())
for p in ADAPTER_DIR.iterdir():
    print("-", p.name)

print("\nCompetition files:")
print("test.csv:", Path(CONFIG["test_path"]).exists(), CONFIG["test_path"])
print("sample_submission.csv:", Path(CONFIG["sample_submission_path"]).exists(), CONFIG["sample_submission_path"])

if FINAL_OFFLINE:
    base_dir = Path(CONFIG["base_model"])
    print("\nBase model local dir exists:", base_dir.exists(), base_dir)
    if base_dir.exists():
        for p in base_dir.iterdir():
            print("-", p.name)

ADAPTER_DIR exists: True
- adapter_model.safetensors
- adapter_config.json
- README.md
- tokenizer.json
- tokenizer_config.json
- chat_template.jinja

Competition files:
test.csv: True /kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/test.csv
sample_submission.csv: True /kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/sample_submission.csv

Base model local dir exists: True /kaggle/input/datasets/chenjunjiejaywang/model-dataset
- model.safetensors.index.json
- config.json
- model-00001-of-00002.safetensors
- model-00002-of-00002.safetensors
- tokenizer.json
- tokenizer_config.json
- chat_template.jinja
- generation_config.json


In [8]:
# ===== CELL 3: LOAD TEST DATA =====
def pick_col(df, candidates, name):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(f"Cannot find {name} column. Available columns: {list(df.columns)}")

test_df = pd.read_csv(CONFIG["test_path"])
sample_sub_df = pd.read_csv(CONFIG["sample_submission_path"])

TEST_ID_COL = pick_col(test_df, ["id"], "id")
TEST_PROMPT_COL = pick_col(test_df, ["prompt", "description", "text", "instruction"], "prompt")

test_df = test_df[[TEST_ID_COL, TEST_PROMPT_COL]].rename(
    columns={TEST_ID_COL: "id", TEST_PROMPT_COL: "prompt"}
).copy()

print("test_df shape:", test_df.shape)
print("sample_sub_df shape:", sample_sub_df.shape)
display(test_df.head())

test_df shape: (1000, 2)
sample_sub_df shape: (1000, 2)


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...
3,8fe82f3af89e487b31236ca829c3f071,The image contains black geometric shapes agai...
4,600464e4d92c75338462271a09b3f176,The image shows a single dark gray triangle po...


In [9]:
# ===== CELL 4: SVG CLEANING UTILS =====
SVG_START_RE = re.compile(r"<svg\b", flags=re.IGNORECASE)
SVG_END_RE = re.compile(r"</svg\s*>", flags=re.IGNORECASE)
XML_DECL_RE = re.compile(r"<\?xml[\s\S]*?\?>", flags=re.IGNORECASE)
DOCTYPE_RE = re.compile(r"<!DOCTYPE[\s\S]*?>", flags=re.IGNORECASE)
COMMENT_RE = re.compile(r"<!--[\s\S]*?-->", flags=re.IGNORECASE)
CODE_FENCE_RE = re.compile(r"```(?:svg)?|```", flags=re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")
NUMBER_RE = re.compile(r"(?<![A-Za-z])[-+]?(?:\d+\.\d+|\d+|\.\d+)(?:[eE][-+]?\d+)?")

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient",
    "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter"
}
VISIBLE_SHAPE_TAGS = {"path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "text", "use"}

def local_name(tag_or_attr):
    s = str(tag_or_attr)
    if "}" in s:
        s = s.split("}", 1)[1]
    if ":" in s:
        s = s.split(":", 1)[1]
    return s

def extract_svg(text):
    if not isinstance(text, str):
        return ""
    text = CODE_FENCE_RE.sub("", text)
    text = XML_DECL_RE.sub("", text)
    text = DOCTYPE_RE.sub("", text)
    text = COMMENT_RE.sub("", text)
    text = text.strip()

    m = SVG_START_RE.search(text)
    if not m:
        return ""

    text = text[m.start():]
    end_matches = list(SVG_END_RE.finditer(text))
    if end_matches:
        text = text[:end_matches[-1].end()]
    return text.strip()

def strip_namespace_inplace(elem):
    elem.tag = local_name(elem.tag)
    new_attrib = {}
    for k, v in elem.attrib.items():
        new_attrib[local_name(k)] = v
    elem.attrib.clear()
    elem.attrib.update(new_attrib)
    for child in list(elem):
        strip_namespace_inplace(child)

def prune_disallowed_children(elem):
    for child in list(elem):
        if local_name(child.tag) not in ALLOWED_TAGS:
            elem.remove(child)
        else:
            prune_disallowed_children(child)

def strip_unsafe_attrs_inplace(elem):
    for attr in list(elem.attrib.keys()):
        name = local_name(attr).lower()
        val = str(elem.attrib[attr]).strip()

        if name.startswith("on"):
            del elem.attrib[attr]
            continue
        if name in {"href", "xlink:href"} and re.match(r"^(https?:)?//", val, flags=re.IGNORECASE):
            del elem.attrib[attr]
            continue

    for child in list(elem):
        strip_unsafe_attrs_inplace(child)

def normalize_numeric_string(x, decimals=2):
    try:
        v = float(x)
    except Exception:
        return x
    if abs(v) < 1e-12:
        v = 0.0
    s = f"{v:.{decimals}f}".rstrip("0").rstrip(".")
    if s == "-0":
        s = "0"
    return s

def compact_path_d(d, decimals=2):
    def repl(m):
        return normalize_numeric_string(m.group(0), decimals=decimals)
    d = NUMBER_RE.sub(repl, str(d))
    d = re.sub(r"\s*,\s*", ",", d)
    d = re.sub(r"\s+", " ", d).strip()
    return d

def compact_attribs_inplace(elem, decimals=2):
    numeric_like = {
        "x", "y", "x1", "y1", "x2", "y2", "cx", "cy", "r", "rx", "ry",
        "width", "height", "stroke-width", "stroke-miterlimit", "opacity",
        "fill-opacity", "stroke-opacity", "offset", "font-size"
    }
    for k in list(elem.attrib.keys()):
        v = str(elem.attrib[k]).strip()
        if k == "d":
            elem.attrib[k] = compact_path_d(v, decimals=decimals)
        elif k in numeric_like:
            elem.attrib[k] = normalize_numeric_string(v, decimals=decimals)
        else:
            elem.attrib[k] = MULTISPACE_RE.sub(" ", v).strip()

    for child in list(elem):
        compact_attribs_inplace(child, decimals=decimals)

def strip_empty_text_inplace(elem):
    if elem.text is not None:
        elem.text = elem.text.strip()
        if elem.text == "":
            elem.text = None
    if elem.tail is not None:
        elem.tail = elem.tail.strip()
        if elem.tail == "":
            elem.tail = None
    for child in list(elem):
        strip_empty_text_inplace(child)

def count_visible_shapes(root):
    return sum(1 for e in root.iter() if local_name(e.tag) in VISIBLE_SHAPE_TAGS)

def minify_svg(svg_text, decimals=2, max_chars=8000):
    svg_text = extract_svg(svg_text)
    if not svg_text:
        return ""

    try:
        root = ET.fromstring(svg_text)
    except Exception:
        return ""

    strip_namespace_inplace(root)
    if local_name(root.tag) != "svg":
        return ""

    prune_disallowed_children(root)
    strip_unsafe_attrs_inplace(root)
    compact_attribs_inplace(root, decimals=decimals)
    strip_empty_text_inplace(root)

    if count_visible_shapes(root) == 0:
        return ""

    root.attrib["xmlns"] = "http://www.w3.org/2000/svg"
    root.attrib["width"] = root.attrib.get("width", "256")
    root.attrib["height"] = root.attrib.get("height", "256")
    root.attrib["viewBox"] = root.attrib.get("viewBox", "0 0 256 256")

    out = ET.tostring(root, encoding="unicode", short_empty_elements=True)
    out = MULTISPACE_RE.sub(" ", out).replace("> <", "><").strip()

    if len(out) > max_chars:
        return ""
    return out

In [10]:
# ===== CELL 5: LOAD MODEL + TOKENIZER =====
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Using dtype:", DTYPE)
print("Loading base model from:", CONFIG["base_model"])

base_infer_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded.")

infer_model = PeftModel.from_pretrained(
    base_infer_model,
    CONFIG["adapter_dir"],
)
infer_model.eval()

print("Adapter loaded.")

# 优先从 adapter 目录读 tokenizer；如果失败再退回 base model
try:
    infer_tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["adapter_dir"],
        trust_remote_code=True,
    )
    print("Tokenizer loaded from adapter dir.")
except Exception:
    infer_tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["base_model"],
        trust_remote_code=True,
    )
    print("Tokenizer loaded from base model.")

if infer_tokenizer.pad_token is None:
    infer_tokenizer.pad_token = infer_tokenizer.eos_token
infer_tokenizer.padding_side = "left"

print("All model assets loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Using dtype: torch.bfloat16
Loading base model from: /kaggle/input/datasets/chenjunjiejaywang/model-dataset


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded.
Adapter loaded.
Tokenizer loaded from adapter dir.
All model assets loaded.


In [11]:
# ===== CELL 6: GENERATION UTILS =====
THINK_RE = re.compile(r"<think>[\s\S]*?</think>", flags=re.IGNORECASE)
CHAT_TAG_RE = re.compile(r"<\|im_start\|>.*?<\|im_end\|>", flags=re.IGNORECASE | re.DOTALL)

FALLBACK_COLORS = {
    "red": "#e53935",
    "blue": "#1e88e5",
    "green": "#43a047",
    "yellow": "#fdd835",
    "orange": "#fb8c00",
    "purple": "#8e24aa",
    "pink": "#d81b60",
    "black": "#111111",
    "white": "#ffffff",
    "gray": "#757575",
    "grey": "#757575",
    "brown": "#6d4c41",
    "gold": "#c9a227",
}

def extract_svg_for_infer(text):
    if not isinstance(text, str):
        return ""
    text = THINK_RE.sub("", text)
    text = CODE_FENCE_RE.sub("", text)
    text = CHAT_TAG_RE.sub("", text)
    text = text.replace("<|im_start|>assistant", "").replace("<|im_end|>", "")
    text = XML_DECL_RE.sub("", text)
    text = DOCTYPE_RE.sub("", text)
    text = COMMENT_RE.sub("", text)
    text = text.strip()

    m = SVG_START_RE.search(text)
    if not m:
        return ""

    text = text[m.start():]
    end_matches = list(SVG_END_RE.finditer(text))
    if end_matches:
        text = text[:end_matches[-1].end()]
    return text.strip()

def sanitize_svg(svg_text):
    return minify_svg(
        extract_svg_for_infer(svg_text),
        decimals=CONFIG["svg_round_decimals"],
        max_chars=CONFIG["max_svg_chars_hard"],
    )

def fallback_svg(prompt):
    p = str(prompt).lower()

    fill = "#111111"
    for k, v in FALLBACK_COLORS.items():
        if k in p:
            fill = v
            break

    if "circle" in p or "dot" in p or "ball" in p:
        shape = f'<circle cx="128" cy="128" r="72" fill="{fill}"/>'
    elif "triangle" in p:
        shape = f'<polygon points="128,48 208,200 48,200" fill="{fill}"/>'
    elif "star" in p:
        shape = f'<polygon points="128,40 149,95 208,95 160,130 178,188 128,153 78,188 96,130 48,95 107,95" fill="{fill}"/>'
    elif "line" in p:
        shape = f'<line x1="48" y1="128" x2="208" y2="128" stroke="{fill}" stroke-width="18" stroke-linecap="round"/>'
    else:
        shape = f'<rect x="56" y="56" width="144" height="144" rx="24" fill="{fill}"/>'

    return f'<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">{shape}</svg>'

BAD_WORDS = ["```", "<think>", "</think>", "<|im_start|>", "<|im_end|>"]
BAD_WORD_IDS = []
for w in BAD_WORDS:
    ids = infer_tokenizer.encode(w, add_special_tokens=False)
    if len(ids) > 0:
        BAD_WORD_IDS.append(ids)

def build_infer_chat(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt)},
    ]
    return infer_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def generate_svg_batch(prompts, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = CONFIG["gen_max_new_tokens"]

    chat_texts = [build_infer_chat(p) for p in prompts]

    inputs = infer_tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    )

    device = next(infer_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            repetition_penalty=CONFIG["repetition_penalty"],
            bad_words_ids=BAD_WORD_IDS,
            renormalize_logits=True,
            pad_token_id=infer_tokenizer.pad_token_id,
            eos_token_id=infer_tokenizer.eos_token_id,
        )

    outputs = []
    for i, prompt in enumerate(prompts):
        gen_ids = output_ids[i, input_width:]
        decoded = infer_tokenizer.decode(gen_ids, skip_special_tokens=True)
        svg = sanitize_svg(decoded)

        if not svg:
            svg = fallback_svg(prompt)
            mode = "fallback"
        else:
            mode = "model"

        outputs.append({"svg": svg, "mode": mode, "raw_text": decoded})

    return outputs

In [12]:
# ===== CELL 7: SMOKE TEST =====
sample_prompts = test_df["prompt"].astype(str).head(8).tolist()
sample_outputs = generate_svg_batch(sample_prompts)

for i, out in enumerate(sample_outputs):
    print("=" * 100)
    print("PROMPT:", sample_prompts[i])
    print("MODE:", out["mode"])
    print("SVG LEN:", len(out["svg"]))
    print(out["svg"][:800])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT: firewood stack cut logs wood with leaf illustration.
MODE: fallback
SVG LEN: 162
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="56" y="56" width="144" height="144" rx="24" fill="#111111"/></svg>
PROMPT: The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.
MODE: fallback
SVG LEN: 193
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><line x1="48" y1="128" x2="208" y2="128" stroke="#ffffff" stroke-width="18" stroke-linecap="round"/></svg>
PROMPT: A stylized icon depicting a curved arrow within a square shape.
MODE: fallback
SVG LEN: 162
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="56" y="56" width="144" height="144" rx="24" fill="#111111"/></svg>
PROMPT: The image contains black geometric shapes against a white background, forming an abstract representation of a person sitting on a 

In [13]:
# ===== CELL 8: BUILD SUBMISSION =====
rows = []
fallback_count = 0
model_count = 0

batch_size = CONFIG["infer_batch_size"]
n = len(test_df)
t0 = time.time()
batch_times = []

for start in range(0, n, batch_size):
    batch_idx = start // batch_size + 1
    batch_t0 = time.time()

    batch_df = test_df.iloc[start:start + batch_size]
    prompts = batch_df["prompt"].astype(str).tolist()
    ids = batch_df["id"].tolist()

    print(
        f"[Batch {batch_idx:04d}] starting | rows={start + 1}-{min(start + batch_size, n)}/{n}",
        flush=True,
    )

    outputs = generate_svg_batch(prompts, max_new_tokens=CONFIG["gen_max_new_tokens"])

    batch_svg_lens = []
    batch_model = 0
    batch_fallback = 0

    for sample_id, out in zip(ids, outputs):
        if out["mode"] == "fallback":
            fallback_count += 1
            batch_fallback += 1
        else:
            model_count += 1
            batch_model += 1

        rows.append({
            "id": sample_id,
            "svg": out["svg"],
        })
        batch_svg_lens.append(len(out["svg"]))

    done = min(start + batch_size, n)
    batch_elapsed = time.time() - batch_t0
    total_elapsed = time.time() - t0
    batch_times.append(batch_elapsed)
    avg_batch_time = sum(batch_times) / len(batch_times)
    rows_left = n - done
    est_min_left = (rows_left / batch_size) * avg_batch_time / 60 if batch_size > 0 else 0

    print(
        f"[Batch {batch_idx:04d}] done={done}/{n} | "
        f"batch_time={batch_elapsed:.2f}s | "
        f"model={batch_model} | "
        f"fallback={batch_fallback} | "
        f"avg_svg_len={sum(batch_svg_lens) / max(1, len(batch_svg_lens)):.1f} | "
        f"elapsed={total_elapsed / 60:.2f}m | "
        f"eta={est_min_left:.2f}m",
        flush=True,
    )

sub_df = pd.DataFrame(rows)

assert list(sub_df.columns) == ["id", "svg"]
assert len(sub_df) == len(test_df)
assert len(sub_df) == len(sample_sub_df)
assert list(sub_df["id"]) == list(sample_sub_df["id"])
assert sub_df["svg"].map(lambda x: isinstance(x, str) and len(x) > 0).all()
assert sub_df["svg"].map(lambda x: len(x) <= CONFIG["max_svg_chars_hard"]).all()

submission_path = Path("/kaggle/working/submission.csv")
sub_df.to_csv(submission_path, index=False)

elapsed_min = (time.time() - t0) / 60
print("\nDone.", flush=True)
print("Saved submission to:", submission_path, flush=True)
print("Rows:", len(sub_df), flush=True)
print("Model count:", model_count, flush=True)
print("Fallback count:", fallback_count, flush=True)
print("Fallback rate:", fallback_count / max(1, len(sub_df)), flush=True)
print(f"Runtime (minutes): {elapsed_min:.2f}", flush=True)

display(sub_df.head())

[Batch 0001] starting | rows=1-12/1000
[Batch 0001] done=12/1000 | batch_time=54.91s | model=2 | fallback=10 | avg_svg_len=213.2 | elapsed=0.92m | eta=75.35m
[Batch 0002] starting | rows=13-24/1000
[Batch 0002] done=24/1000 | batch_time=56.86s | model=1 | fallback=11 | avg_svg_len=188.9 | elapsed=1.86m | eta=75.75m
[Batch 0003] starting | rows=25-36/1000
[Batch 0003] done=36/1000 | batch_time=58.74s | model=2 | fallback=10 | avg_svg_len=197.5 | elapsed=2.84m | eta=76.10m
[Batch 0004] starting | rows=37-48/1000
[Batch 0004] done=48/1000 | batch_time=59.70s | model=3 | fallback=9 | avg_svg_len=235.4 | elapsed=3.84m | eta=76.10m
[Batch 0005] starting | rows=49-60/1000
[Batch 0005] done=60/1000 | batch_time=58.11s | model=2 | fallback=10 | avg_svg_len=198.4 | elapsed=4.81m | eta=75.28m
[Batch 0006] starting | rows=61-72/1000
[Batch 0006] done=72/1000 | batch_time=58.54s | model=2 | fallback=10 | avg_svg_len=199.7 | elapsed=5.78m | eta=74.51m
[Batch 0007] starting | rows=73-84/1000
[Batch 0

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg viewBox=""0.0 0.0 200.0 200.0"" height=""200..."


In [14]:
# ===== CELL 9: FINAL CHECK =====
submission_path = Path("/kaggle/working/submission.csv")
print("submission exists:", submission_path.exists())
print("submission size (MB):", submission_path.stat().st_size / 1024 / 1024 if submission_path.exists() else None)

check_df = pd.read_csv(submission_path)
print(check_df.shape)
display(check_df.head())

submission exists: True
submission size (MB): 0.2348613739013672
(1000, 2)


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg viewBox=""0.0 0.0 200.0 200.0"" height=""200..."
